# Mortgage Credit Risk Analysis & Macro Stress Testing
### End-to-End Quantitative Risk Analytics on 250,000 Freddie Mac Mortgages

**Author:** Naman Roy  
**Role:** Data Analyst / Credit Risk Analytics  
**Portfolio Target:** Conforming Single-Family Mortgages (Vintages 2012, 2016, 2019, 2021, 2023)  
**Macro Horizon:** Federal Reserve 2026 Severely Adverse Scenario

## 1. Executive Summary & Problem Definition

This study evaluates credit risk across a $25.8B mortgage portfolio comprising 250,000 Freddie Mac loans. By integrating high-performance SQL in DuckDB with statistical credit risk modeling, we answer five core portfolio risk questions:

1. **Cohort Vintage Degradation:** How do vintage delinquency trajectories compare across loan seasoning ages?
2. **Macroeconomic Shocks vs Underwriting:** Was the COVID-19 delinquency spike a structural underwriting issue or an exogenous calendar shock?
3. **State Transition Dynamics:** How do loans transition across delinquency stages (roll-rate matrix) under benign vs stressed regimes?
4. **Statistical PD Modeling:** Can a loan-level logistic regression scorecard outperform traditional two-dimensional segment PD tables?
5. **Severely Adverse Capital Impact:** Under the Federal Reserve's 2026 Severe Scenario (-30% HPI, 10% unemployment), how much capital reserve is required, and where does loss concentrate?

In [1]:
import os
from pathlib import Path
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# Ensure consistent style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
OUT = Path("outputs")
DB = "mortgage.duckdb"

## 2. Key Portfolio Risk Indicators (KPIs)

In [2]:
kpis = pd.DataFrame([
    ("Active Book Balance", "$25.8 Billion", "114,860 Active Mortgages as of Mar 2026"),
    ("90+ Days Late Rate", "0.75%", "2.1x increase vs two years ago (0.36%)"),
    ("2023 Vintage 90+ Late", "1.24%", "Highest of all vintages; up from 0.87% YoY"),
    ("12-Month Expected Loss", "$12.5M (4.9 bps)", "Scorecard PD adjusted; 60% from 2023 vintage"),
    ("Fed Severe Stressed Loss", "$128.6M (49.8 bps)", "11.0x baseline; 52% of 2023 loans fall underwater"),
    ("LGD Loss Severity", "23.7%", "30.8% judicial states vs 19.6% non-judicial"),
    ("Scorecard Discrimination", "AUC 0.90 / Gini 0.80", "KS 0.64; Top decile holds 79% of defaults"),
], columns=["Metric", "Value", "Credit Risk Context"])

kpis

## 3. Delinquency Trajectories by Vintage (Loan Age)

In [3]:
vc = pd.read_csv(OUT / "vintage_curves.csv")
display(vc.groupby("vintage")["cum_d90_pct"].max().to_frame("Max_Cumulative_D90_Pct"))

# Display generated high-resolution vintage curve plot
if (OUT / "vintage_curves.png").exists():
    from IPython.display import Image
    display(Image(filename=str(OUT / "vintage_curves.png")))

## 4. Calendar Delinquency & The COVID Shock

In [4]:
if (OUT / "calendar_d90.png").exists():
    display(Image(filename=str(OUT / "calendar_d90.png")))

cal = pd.read_csv(OUT / "calendar_d90.csv")
cal['month'] = pd.to_datetime(cal['month'])
covid_peak = cal[(cal['month'] >= '2020-03-01') & (cal['month'] <= '2020-12-01')]
print("Peak Delinquency Month During COVID:")
display(covid_peak.groupby('month')['d90_rate_pct'].mean().nlargest(3).to_frame('Avg_D90_Pct'))

## 5. Delinquency Transition Dynamics (Roll Rates)

In [5]:
rr = pd.read_csv(OUT / "roll_rate.csv")
print("Month-over-Month State Transitions: Normal vs COVID Regime (%)")
display(rr.pivot(index=["period", "from_state"], columns="to_state", values="pct"))

## 6. Risk Drivers & The FICO x LTV Matrix

In [6]:
if (OUT / "risk_drivers_lift.png").exists():
    display(Image(filename=str(OUT / "risk_drivers_lift.png")))

if (OUT / "risk_grid_heatmap.png").exists():
    display(Image(filename=str(OUT / "risk_grid_heatmap.png")))

## 7. Statistical PD Scorecard & Out-of-Time Backtesting

In [7]:
metrics = pd.read_csv(OUT / "pd_model_metrics.csv")
print("--- Model Discrimination & Out-of-Time Validation ---")
display(metrics)

if (OUT / "pd_calibration.png").exists():
    display(Image(filename=str(OUT / "pd_calibration.png")))
if (OUT / "pd_backtest_time.png").exists():
    display(Image(filename=str(OUT / "pd_backtest_time.png")))

## 8. Expected Loss & Federal Reserve Stress Testing Waterfall

In [8]:
stress_wf = pd.read_csv(OUT / "stress_waterfall.csv")
display(stress_wf)

if (OUT / "stress_waterfall.png").exists():
    display(Image(filename=str(OUT / "stress_waterfall.png")))
if (OUT / "stress_vintage.png").exists():
    display(Image(filename=str(OUT / "stress_vintage.png")))

## 9. Strategic Credit Recommendations

1. **Tighten Underwriting Cutoffs on High LTV / High DTI Layering:**
   - Limit approval of mortgages where LTV > 90% combined with DTI > 43%.
   - FICO scores below 660 with LTV > 80% demonstrate a default multiplier exceeding 5.5x.
2. **Implement Early-Intervention for the 2023 Vintage:**
   - Proactive outreach at 30-day delinquency reduces 30->60 roll risk significantly.
   - 2023 borrowers carry minimal home equity cushions and elevated monthly interest burdens.
3. **Capital Reserve Allocation:**
   - Baseline credit reserve of $12.5M must be buffered by an additional $116M CCAR stress reserve.